# agent

> Restricted execution for AI agents over the owner's live namespace: an overlay layer
> (agents read everything, create their own names, and can never rebind or delete the
> owner's) plus a clikernel-compatible inspector hook — an AST restriction list checked
> before every cell runs.

In [ ]:
#| default_exp agent

In [ ]:
#| hide
from nbdev.showdoc import *

## Inspectors

The inspector contract is [clikernel](https://github.com/AnswerDotAI/clikernel)-compatible, so one
rules file can serve both a clikernel worker and a paar agent session: an inspector is a function
taking the cell AST (and optionally the raw source) that returns a note string to prepend to the
output, returns `None` to pass, or raises `RuleBlock` to refuse the cell. Rules are steering for
well-behaved agents, **not** a security sandbox.

In [ ]:
#| export
import ast, os, runpy, inspect
from pathlib import Path
from IPython.utils.capture import capture_output
from paar.runtime import ExecResult, _exec_capture, _flat_result
from paar.snapshot import snapshot, expand, grid_page, profile_view
from paar.bridge import _ns

class RuleBlock(Exception):
    "Raised by an inspector to deliberately block a cell; any other inspector exception is a bug and fails open (clikernel-compatible)."

In [ ]:
#| export
def load_inspectors(path=None):
    "Inspector functions from `path`, $PAAR_INSPECTORS, or ~/.config/paar/inspectors.py; the file may define `inspect(tree[, code])` and/or a list `inspectors` (clikernel's file contract)."
    p = path or os.environ.get('PAAR_INSPECTORS') or Path(os.environ.get('XDG_CONFIG_HOME', str(Path.home()/'.config')))/'paar'/'inspectors.py'
    p = Path(p).expanduser()
    if not p.exists(): return []
    ns = runpy.run_path(str(p))
    ins = list(ns.get('inspectors', []))
    if callable(ns.get('inspect')): ins.append(ns['inspect'])
    return ins

def _call_inspector(f, tree, code):
    "1-arg inspectors get the transformed AST; 2-arg ones also get the raw cell source."
    return f(tree, code) if len(inspect.signature(f).parameters) > 1 else f(tree)

def run_inspectors(inspectors, code):
    "Concatenated notes from every inspector for `code` ('' for unparseable code — execution reports the syntax error); RuleBlock propagates, any other inspector exception fails open with a warning note."
    try: tree = ast.parse(code)
    except SyntaxError: return ''
    notes = []
    for f in inspectors:
        try: n = _call_inspector(f, tree, code)
        except RuleBlock: raise
        except Exception as e: n = f'[inspector {getattr(f, "__name__", "?")} crashed, check skipped: {type(e).__name__}: {e}]\n'
        if n: notes.append(n)
    return ''.join(notes)

## The default restriction list

`policy_inspectors` implements the agent contract: read anything, create anything new, never
touch what the owner made. "Owner names" are computed per call, so anything the agent has
shadowed in its own layer is its to mutate. Destructive filesystem/shell escapes are blocked
outright — an agent has its own (permission-gated) shell; the owner's kernel is not it.

In [ ]:
#| export
_MUTATORS = frozenset('append extend insert remove pop clear update add discard popitem setdefault sort reverse'.split())
_BLOCKED_CALLS = frozenset('exec eval setattr delattr globals vars breakpoint'.split())
_DESTRUCTIVE = {('os','remove'),('os','unlink'),('os','rmdir'),('os','removedirs'),('os','rename'),
                ('os','replace'),('os','system'),('os','popen'),('shutil','rmtree'),('shutil','move')}
_DESTRUCTIVE_ATTRS = frozenset(('rmtree','unlink','rmdir'))

def _root(node):
    "Base Name id of a dotted/indexed/called chain (x.a[0].b() -> 'x'), or None."
    while isinstance(node, (ast.Attribute, ast.Subscript, ast.Call)):
        node = node.func if isinstance(node, ast.Call) else node.value
    return node.id if isinstance(node, ast.Name) else None

def _block(msg): raise RuleBlock(msg)

def _check_del(node, owner):
    for t in node.targets:
        r = t.id if isinstance(t, ast.Name) else _root(t)
        if r in owner: _block(f"cannot delete owner variable '{r}'; you may only delete names you created")

def _check_assign(node, owner):
    tgts = node.targets if isinstance(node, ast.Assign) else [node.target]
    for t in tgts:
        for u in ast.walk(t):   # covers tuple/list unpacking targets
            if isinstance(u, (ast.Attribute, ast.Subscript)):
                r = _root(u)
                if r in owner: _block(f"cannot modify owner variable '{r}' in place; bind your result to a new name instead")
    if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name) and node.target.id in owner:
        _block(f"'{node.target.id} += ...' may mutate the owner's object in place; use a new name, e.g. {node.target.id}2 = {node.target.id} + ...")

def _check_call(node, owner):
    f = node.func
    if isinstance(f, ast.Name) and f.id in _BLOCKED_CALLS:
        _block(f"call to '{f.id}' is not allowed in the agent session")
    if isinstance(f, ast.Attribute):
        r = _root(f.value)
        if r == 'subprocess' or (isinstance(f.value, ast.Name) and (f.value.id, f.attr) in _DESTRUCTIVE):
            _block(f"'{r}.{f.attr}' is a shell/filesystem escape and is not allowed in the agent session; use your own shell tool")
        if f.attr in _DESTRUCTIVE_ATTRS:
            _block(f"'.{f.attr}()' deletes resources and is not allowed in the agent session")
        if f.attr in _MUTATORS and r in owner:
            _block(f"'.{f.attr}()' would mutate owner variable '{r}'; copy it to a new name first")
        if r in owner and any(k.arg == 'inplace' and getattr(k.value, 'value', None) is True for k in node.keywords):
            _block(f"inplace=True would mutate owner variable '{r}'; drop inplace and bind the result to a new name")

def policy_inspectors(owner_names):
    "The default restriction list, closed over `owner_names` (a callable giving the currently protected names): agents may read owner variables and create their own; deleting, rebinding in place, or mutating owner state — and destructive file/shell escapes — are blocked."
    def guard(tree, code):
        owner = owner_names()
        for node in ast.walk(tree):
            if isinstance(node, ast.Delete): _check_del(node, owner)
            elif isinstance(node, (ast.Assign, ast.AugAssign, ast.AnnAssign)): _check_assign(node, owner)
            elif isinstance(node, ast.Call): _check_call(node, owner)
            elif isinstance(node, ast.Attribute) and node.attr in ('__dict__', '__globals__') and _root(node) in owner:
                _block(f"direct '{node.attr}' access on owner objects is not allowed")
        return None
    return [guard]

## AgentSession

The overlay: `ns` is the agent's persistent exec globals — the owner's namespace refreshed in
under the agent's own `layer` before each run, and diffed back into `layer` afterward. The owner
dict itself is never written, so rebinding an owner name merely shadows it and deleting one is
impossible by construction (and blocked by the policy for a clear error). `ns` keeps a stable
identity so functions the agent defines keep live globals across calls.

In [ ]:
#| export
class AgentSession:
    "A persistent agent overlay on the owner namespace: reads fall through, writes land in the agent's own layer, and every cell passes the restriction list first."
    def __init__(self,
                 owner=None,   # callable -> namespace dict to protect (default: paar's inspected namespace)
                 rules=None):  # extra inspectors appended to the default policy (default: load_inspectors())
        self.owner = owner if owner is not None else _ns
        self.layer = {}   # names the agent created (or shadowed)
        self.ns = {}      # persistent exec globals: owner view + layer
        self.inspectors = policy_inspectors(self.owner_names) + (load_inspectors() if rules is None else list(rules))

    def owner_names(self):
        "Currently protected names: everything in the owner namespace the agent hasn't shadowed."
        return {k for k in self.owner() if k not in self.layer}

    def _sync_in(self):
        owner = self.owner()
        for k, v in owner.items():
            if k not in self.layer: self.ns[k] = v
        for k in [k for k in self.ns if k not in owner and k not in self.layer and k != '__builtins__']:
            del self.ns[k]   # owner deleted it; the agent's view follows
        return self.ns

    def _sync_out(self):
        owner = self.owner()
        for k, v in self.ns.items():
            if k != '__builtins__' and owner.get(k) is not v: self.layer[k] = v
        for k in [k for k in self.layer if k not in self.ns]: del self.layer[k]

    def view_ns(self):
        "Merged read view (owner + agent layer) for snapshots/expands/grids."
        return {k: v for k, v in self._sync_in().items() if k != '__builtins__'}

    def run(self, code, scope='overlay'):
        "Run agent code under the restriction list. scope='overlay' persists new names in the agent layer; 'isolated' discards all writes. The owner namespace is never written either way."
        try: note = run_inspectors(self.inspectors, code)
        except RuleBlock as e: return ExecResult(ok=False, error=f'blocked: {e}')
        run_ns = self._sync_in() if scope == 'overlay' else dict(self._sync_in())
        cap = err = val = None
        try:
            with capture_output() as cap: val = _exec_capture(code, run_ns)
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
        if scope == 'overlay': self._sync_out()
        out = note + (cap.stdout if cap is not None else '')
        if err: return ExecResult(ok=False, stdout=out, error=err)
        return ExecResult(ok=True, result=None if val is None else _flat_result(val), stdout=out)

    def snapshot(self, name=None, typ=None): return snapshot(self.view_ns(), frozenset(), name, typ)
    def view(self, profile='standard'): return profile_view(self.view_ns(), frozenset(), profile)
    def expand(self, accessor, offset=0): return expand(self.view_ns(), accessor, offset)
    def grid(self, accessor, roff=0, coff=0, rows=50, cols=50): return grid_page(self.view_ns(), accessor, roff, coff, rows, cols)

## Tests

In [ ]:
owner = {'x': 1, 'lst': [1, 2], 'd': {'a': 1}}
s = AgentSession(owner=lambda: owner, rules=[])

def test_create_and_read():
    r = s.run('a = x + 41'); assert r.ok, r.error
    assert s.layer['a'] == 42 and 'a' not in owner
    r = s.run('a'); assert r.ok and r.result.value == '42'

def test_shadow_not_clobber():
    r = s.run('x = 99'); assert r.ok, r.error
    assert owner['x'] == 1 and s.layer['x'] == 99
    assert s.run('x').result.value == '99'   # agent sees its shadow
    assert 'x' not in s.owner_names()        # and may now treat it as its own

def test_del_and_mutation_blocked():
    for bad in ('del lst', 'lst.append(3)', 'lst[0] = 5', 'lst += [4]',
                "d.update(b=2)", "d['b'] = 2", 'lst.sort()'):
        r = s.run(bad)
        assert not r.ok and r.error.startswith('blocked:'), (bad, r.error)
    assert owner['lst'] == [1, 2] and owner['d'] == {'a': 1}

def test_own_copies_are_mutable():
    assert s.run('mine = list(lst)').ok
    r = s.run('mine.append(3); mine'); assert r.ok, r.error
    assert s.layer['mine'] == [1, 2, 3] and owner['lst'] == [1, 2]

def test_escapes_blocked():
    for bad in ('exec("del lst")', 'eval("lst.clear()")', 'globals()', 'setattr(d, "x", 1)',
                'import shutil; shutil.rmtree("/nope")', 'import subprocess; subprocess.run("ls")',
                'import os; os.remove("/nope")', 'from pathlib import Path; Path("/nope").unlink()'):
        r = s.run(bad)
        assert not r.ok and r.error.startswith('blocked:'), (bad, r.error)

def test_isolated_discards():
    r = s.run('tmp_iso = 5', scope='isolated'); assert r.ok
    assert 'tmp_iso' not in s.layer and 'tmp_iso' not in s.ns

def test_agent_can_del_own():
    assert s.run('tmp = 1').ok and 'tmp' in s.layer
    assert s.run('del tmp').ok and 'tmp' not in s.layer

def test_functions_keep_live_globals():
    assert s.run('def f(): return q').ok
    assert s.run('q = 7').ok
    r = s.run('f()'); assert r.ok and r.result.value == '7'

def test_owner_deletion_followed():
    owner['gone'] = 3
    assert s.run('gone').ok
    del owner['gone']
    r = s.run('gone'); assert not r.ok and 'NameError' in r.error

for t in [v for k, v in list(globals().items()) if k.startswith('test_')]: t()

In [ ]:
# custom rules: a RuleBlock inspector blocks, a crashing inspector fails open with a note
def no_imports(tree):
    if any(isinstance(n, (ast.Import, ast.ImportFrom)) for n in ast.walk(tree)):
        raise RuleBlock('imports are disabled here')

def buggy(tree): raise ValueError('oops')

s2 = AgentSession(owner=lambda: {}, rules=[no_imports, buggy])
r = s2.run('import math'); assert not r.ok and 'imports are disabled' in r.error
r = s2.run('v = 1');       assert r.ok and 'crashed, check skipped' in r.stdout and s2.layer['v'] == 1

In [ ]:
# load_inspectors: clikernel-style rules file (an `inspect` fn and an `inspectors` list)
import tempfile
with tempfile.TemporaryDirectory() as td:
    p = Path(td)/'inspectors.py'
    p.write_text("import ast\n"
                 "def inspect(tree):\n"
                 "    if any(isinstance(n, ast.While) for n in ast.walk(tree)): return 'note: a while loop\\n'\n"
                 "inspectors = []\n")
    ins = load_inspectors(p)
    assert len(ins) == 1
    s3 = AgentSession(owner=lambda: {}, rules=ins)
    r = s3.run('i = 0\nwhile i < 3: i += 1')
    assert r.ok and r.stdout.startswith('note: a while loop')
assert load_inspectors(Path(td)/'missing.py') == []

## Watching and extending

The owner sees everything: `paar.fasthtml.agent_layer()` returns the live dict of names the
agent has created — assign it to a variable and it appears, expandable, in your own panel.
To tighten or loosen the policy, ship extra rules in `~/.config/paar/inspectors.py` (or
point `$PAAR_INSPECTORS` at a file); because the contract matches clikernel's, the same
file can guard an agent's own clikernel worker. Passing `rules=[...]` to `AgentSession`
overrides file loading entirely (the built-in policy always applies).

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()